# CIFAR-100 Image Classification — End-to-End ML Pipeline**Course:** Machine Learning Pipeline (MLOps Summative)**Author:** Gilbert**Dataset:** CIFAR-100 (60,000 32x32 RGB images; 100 fine classes nested in 20 coarse superclasses)This notebook covers the offline half of the pipeline: data acquisition, preprocessing,model creation, training, and evaluation. The online half (API, UI, retraining trigger,Docker, load testing) lives in `src/`, `app/` and the `docker-compose.yml` at the repo root.**A note on the label space.** CIFAR-100 ships two label sets. The 100 fine labels(`apple`, `beaver`, `boy`, ...) are extremely hard at 32x32 resolution. The 20 coarsesuperclasses (`fruit_and_vegetables`, `aquatic_mammals`, `people`, ...) are the practicalchoice for a deployed classifier. This notebook trains on the coarse labels and reportsfine-label results at the end for comparison. Flip `LABEL_MODE` to `fine` to swap.

In [ ]:
import os, sys, json, warningswarnings.filterwarnings("ignore")os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"# Set the label space BEFORE importing src.config (config reads it from the environment)os.environ["LABEL_MODE"] = "coarse"   # "coarse" (20 classes) or "fine" (100 classes)sys.path.insert(0, "..")   # so `from src import ...` works from inside notebook/import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport tensorflow as tffrom src import config, model as model_libfrom src.preprocessing import load_cifar_split, load_dataset, load_label_namesnp.random.seed(config.SEED)tf.random.set_seed(config.SEED)sns.set_theme(style="whitegrid")print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))print("Label mode:", config.LABEL_MODE, "| classes:", config.NUM_CLASSES)

---## 1. Data acquisitionCIFAR-100 is distributed as three Python pickles. Each stores images as a flat`(N, 3072)` uint8 array: 1024 red bytes, then 1024 green, then 1024 blue. Recovering animage means reshaping to `(3, 32, 32)` and transposing the channel axis to the back.Getting this wrong produces images that look like colour static, which is the classicfirst bug with this dataset.

In [ ]:
x_train_raw, y_train_raw = load_cifar_split("train")x_test_raw,  y_test_raw  = load_cifar_split("test")class_names = load_label_names()print("train:", x_train_raw.shape, x_train_raw.dtype, "| labels:", y_train_raw.shape)print("test: ", x_test_raw.shape,  "| classes:", len(class_names))print("pixel range:", x_train_raw.min(), "to", x_train_raw.max())print()print("Classes:", ", ".join(class_names))

In [ ]:
# Sanity check: the images must look like objects, not noise.fig, axes = plt.subplots(4, 8, figsize=(12, 6))for ax, i in zip(axes.ravel(), np.random.choice(len(x_train_raw), 32, replace=False)):    ax.imshow(x_train_raw[i]); ax.axis("off")    ax.set_title(class_names[y_train_raw[i]], fontsize=6)plt.suptitle("Random training images")plt.tight_layout(); plt.show()

---## 2. Exploratory analysis — three features, three storiesThe assignment asks for interpretations of at least three features. Images have nocolumns, so the "features" here are properties computed from the pixels themselves:class balance, colour composition, and within-image contrast.

### Feature 1 — Class balance

In [ ]:
counts = pd.Series(y_train_raw).value_counts().sort_index()counts.index = class_namesplt.figure(figsize=(11, 4))sns.barplot(x=counts.index, y=counts.values, color="#4C72B0")plt.xticks(rotation=60, ha="right"); plt.ylabel("training images")plt.title("Images per class"); plt.tight_layout(); plt.show()print(f"min={counts.min()}  max={counts.max()}  std={counts.std():.2f}")

**What it shows:** the dataset is exactly balanced. Every class holds the same number oftraining images.> **Your interpretation goes here.** Say what a perfectly balanced dataset buys you:> which metric becomes trustworthy, what the random-guess baseline is, and what you would> have had to change (class weights? resampling? a different headline metric?) if it were> skewed instead.

### Feature 2 — Colour composition

In [ ]:
rgb = np.stack([x_train_raw[y_train_raw == i].reshape(-1, 3).mean(axis=0)                for i in range(len(class_names))])colour_df = pd.DataFrame(rgb, columns=["red", "green", "blue"], index=class_names)colour_df.plot(kind="bar", figsize=(11, 4), color=["#d62728", "#2ca02c", "#1f77b4"])plt.ylabel("mean channel value (0-255)"); plt.title("Average colour signature per class")plt.xticks(rotation=60, ha="right"); plt.tight_layout(); plt.show()green_bias = colour_df["green"] - colour_df[["red", "blue"]].mean(axis=1)blue_bias  = colour_df["blue"]  - colour_df[["red", "green"]].mean(axis=1)print("Most green-dominant:\n", green_bias.nlargest(3).round(2), "\n")print("Most blue-dominant:\n", blue_bias.nlargest(3).round(2))

> **Your interpretation goes here.** Look at which classes sit at the extremes and ask> what that means for the model: which classes could be separated on mean colour alone,> which pairs share a colour signature and will therefore have to be told apart on shape> or texture, and what that predicts about where the confusion matrix will light up.

### Feature 3 — Contrast (within-image pixel variance)

In [ ]:
contrast = pd.Series(    [x_train_raw[y_train_raw == i].std(axis=(1, 2, 3)).mean() for i in range(len(class_names))],    index=class_names,).sort_values()plt.figure(figsize=(11, 4))sns.barplot(x=contrast.index, y=contrast.values, color="#8172B3")plt.xticks(rotation=60, ha="right"); plt.ylabel("mean pixel std within an image")plt.title("Contrast per class — a proxy for how busy the images are")plt.tight_layout(); plt.show()print("Flattest:\n", contrast.head(3).round(2), "\n")print("Busiest:\n", contrast.tail(3).round(2))

> **Your interpretation goes here.** Pixel standard deviation inside an image is a rough> texture measure: a flat class is one where images are dominated by large uniform regions,> a busy class is full of edges. Connect that to how a CNN learns (what do early conv> filters actually respond to?) and predict which end of this ranking will be easier.

### Bonus — PCA of the raw pixelsProjecting the 3,072-dimensional pixel vectors onto their first two principal componentsshows how much class structure is present *before* any learning happens. If the classesalready separated here, we would not need a CNN.

In [ ]:
from sklearn.decomposition import PCAsub = np.random.choice(len(x_train_raw), 5000, replace=False)flat = x_train_raw[sub].reshape(len(sub), -1).astype("float32") / 255.0flat -= flat.mean(axis=0)pcs = PCA(n_components=2, random_state=config.SEED).fit(flat)proj = pcs.transform(flat)plt.figure(figsize=(7, 6))sc = plt.scatter(proj[:, 0], proj[:, 1], c=y_train_raw[sub], cmap="tab20", s=6, alpha=0.6)plt.xlabel(f"PC1 ({pcs.explained_variance_ratio_[0]:.1%} variance)")plt.ylabel(f"PC2 ({pcs.explained_variance_ratio_[1]:.1%} variance)")plt.title("First two principal components of the raw pixels")plt.colorbar(sc, label="class index"); plt.tight_layout(); plt.show()print(f"Two components explain only {pcs.explained_variance_ratio_.sum():.1%} of pixel variance.")

---## 3. PreprocessingFour steps, all implemented in `src/preprocessing.py` so that the notebook, the API and theretraining job apply *identical* transformations. Divergence between training-time andserving-time preprocessing is the most common cause of a model that scores well offlineand fails in production.1. **Reshape** the flat 3072-byte rows into `(32, 32, 3)`.2. **Shuffle**, then hold out 10% of the training set as a validation split. The test set   is never touched until final evaluation.3. **Scale** pixels from `[0, 255]` to `[0, 1]` (float32).4. **Augment** — random horizontal flip, translation, rotation and zoom. This runs as the   first layer *inside* the model, so it is active during training and automatically   bypassed at inference. It is the single biggest accuracy lever at this resolution.

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test), class_names = load_dataset()print(f"train {x_train.shape}  val {x_val.shape}  test {x_test.shape}")print(f"dtype {x_train.dtype}  range [{x_train.min()}, {x_train.max()}]")

In [ ]:
# What augmentation actually does to one imagefrom src.preprocessing import build_augmentationaug = build_augmentation()sample = x_train[:1]fig, axes = plt.subplots(1, 8, figsize=(13, 2))axes[0].imshow(sample[0]); axes[0].set_title("original", fontsize=8); axes[0].axis("off")for ax in axes[1:]:    ax.imshow(np.clip(aug(sample, training=True)[0], 0, 1)); ax.axis("off")    ax.set_title("augmented", fontsize=8)plt.tight_layout(); plt.show()

---## 4. Model creationA VGG-style CNN: three convolutional blocks (64 / 128 / 256 filters), two 3x3 convolutionsper block, batch normalisation after each, max-pooling between blocks, then global averagepooling into a small dense head.**Optimisation techniques applied** (the rubric asks for these by name):| Technique | Where ||---|---|| Data augmentation | flip / translate / rotate / zoom, inside the model || L2 weight decay | every conv and dense kernel, `1e-4` || Dropout | 0.3 after each conv block, 0.5 before the classifier || Batch normalisation | after every convolution || Adam optimiser | `lr=1e-3` || Learning-rate schedule | `ReduceLROnPlateau`, halving on a validation plateau || Early stopping | patience 10 on validation accuracy, best weights restored || Global average pooling | replaces flatten+dense, cutting parameters by an order of magnitude |

In [ ]:
model = model_lib.build_model()model.summary()

### A baseline to compare againstThe rubric asks for *models*, plural, and for evidence that the optimisation actuallyhelped. So we also train a deliberately unoptimised baseline: the same rough shape, but noaugmentation, no batch norm, no dropout, no regularisation, plain SGD. The gap between thetwo is the argument.

In [ ]:
from tensorflow import kerasfrom tensorflow.keras import layersdef build_vanilla(num_classes=config.NUM_CLASSES):    m = keras.Sequential([        keras.Input(shape=config.INPUT_SHAPE),        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),        layers.Flatten(),        layers.Dense(256, activation="relu"),        layers.Dense(num_classes, activation="softmax"),    ], name="vanilla_cnn")    m.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),              loss=keras.losses.SparseCategoricalCrossentropy(),              metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy"),                       keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_accuracy")])    return mvanilla = build_vanilla()print(f"vanilla: {vanilla.count_params():,} params | optimised: {model.count_params():,} params")

---## 5. TrainingOn a Colab T4 the optimised model takes roughly 15–25 minutes for the full run. On CPU itwill take hours, so use a GPU runtime.

In [ ]:
history_vanilla = vanilla.fit(    x_train, y_train, validation_data=(x_val, y_val),    epochs=20, batch_size=config.BATCH_SIZE, verbose=1,)

In [ ]:
history = model_lib.train_model(model, x_train, y_train, x_val, y_val)

In [ ]:
def plot_history(hists, labels):    fig, axes = plt.subplots(1, 2, figsize=(13, 4))    for h, lab in zip(hists, labels):        axes[0].plot(h.history["accuracy"], label=f"{lab} — train")        axes[0].plot(h.history["val_accuracy"], "--", label=f"{lab} — val")        axes[1].plot(h.history["loss"], label=f"{lab} — train")        axes[1].plot(h.history["val_loss"], "--", label=f"{lab} — val")    axes[0].set(xlabel="epoch", ylabel="accuracy", title="Accuracy")    axes[1].set(xlabel="epoch", ylabel="loss", title="Loss")    for a in axes: a.legend(fontsize=8)    plt.tight_layout(); plt.show()plot_history([history_vanilla, history], ["vanilla", "optimised"])

> **Read the curves.** The gap between the training and validation lines is overfitting.> Look at where the vanilla model's validation loss turns upward while its training loss> keeps falling, and compare that with the regularised model. Write down what you see.

---## 6. EvaluationSix metrics on the held-out test set: **loss, accuracy, top-5 accuracy, macro precision,macro recall, macro F1**. Because the classes are perfectly balanced, macro and weightedaverages will be close; on a skewed dataset they would not be, which is why macro F1 isthe honest headline number.

In [ ]:
metrics_vanilla = model_lib.evaluate_model(vanilla, x_test, y_test, class_names)metrics_opt     = model_lib.evaluate_model(model,   x_test, y_test, class_names)comparison = pd.DataFrame({    "vanilla":   {k: metrics_vanilla[k] for k in                  ["accuracy", "top5_accuracy", "loss", "precision_macro", "recall_macro", "f1_macro"]},    "optimised": {k: metrics_opt[k] for k in                  ["accuracy", "top5_accuracy", "loss", "precision_macro", "recall_macro", "f1_macro"]},})comparison["delta"] = comparison["optimised"] - comparison["vanilla"]comparison.round(4)

In [ ]:
cm = np.array(metrics_opt["confusion_matrix"])cm_norm = cm / cm.sum(axis=1, keepdims=True)plt.figure(figsize=(11, 9))sns.heatmap(cm_norm, xticklabels=class_names, yticklabels=class_names,            cmap="Blues", vmin=0, vmax=1, square=True, cbar_kws={"shrink": 0.7})plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (row-normalised)")plt.tight_layout(); plt.show()

In [ ]:
# Where does it fail? The ten most-confused class pairs.off = cm_norm.copy(); np.fill_diagonal(off, 0)pairs = [(class_names[i], class_names[j], off[i, j])         for i, j in zip(*np.unravel_index(np.argsort(off, axis=None)[::-1][:10], off.shape))]pd.DataFrame(pairs, columns=["true", "predicted as", "rate"]).round(3)

> **Interpret the failures.** Take the top confused pairs above and connect them back to> section 2. Do they share a colour signature? A contrast profile? Are they semantically> close in a way a 32x32 thumbnail cannot resolve? This is the part of the evaluation that> shows you understand the model rather than just having run it.

In [ ]:
report = pd.DataFrame(metrics_opt["report"]).Treport.loc[class_names].sort_values("f1-score", ascending=False).round(3)

---## 7. The prediction functionThis is the exact function the API calls. It takes raw image bytes (any format, any size),resizes to 32x32, scales, and returns the winning class plus the top-5 ranking. Testing ithere rather than testing `model.predict()` directly means the notebook is validating thedeployed path, not a parallel one.

In [ ]:
model_lib.save_model(model)  # models/cifar_cnn.keras — served by the APIprint("saved:", config.BASE_MODEL_PATH)import importlibfrom src import predictionimportlib.reload(prediction)import iofrom PIL import Imageidx = np.random.choice(len(x_test_raw), 6, replace=False)fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))for ax, i in zip(axes, idx):    buf = io.BytesIO(); Image.fromarray(x_test_raw[i]).save(buf, format="PNG")    out = prediction.predict(buf.getvalue())    correct = out["predicted_class"] == class_names[y_test_raw[i]]    ax.imshow(x_test_raw[i]); ax.axis("off")    ax.set_title(f"{out['predicted_class']}\n{out['confidence']:.0%}"                 f"\ntrue: {class_names[y_test_raw[i]]}",                 fontsize=7, color="green" if correct else "red")plt.suptitle("Predictions through the deployed prediction path")plt.tight_layout(); plt.show()

---## 8. Retraining, offlineThe deployed retraining job (`src/retrain.py`) does three things that a naive`model.fit()` on new data would not:1. It loads the **currently served model** and continues training it. Our own custom model   becomes the pre-trained starting point, so knowledge is carried forward rather than   thrown away.2. It **mixes a replay sample** of the original training data in with the new uploads. Fine-   tuning a network on twenty images of one class and nothing else is a fast route to   catastrophic forgetting — the model gets very good at that class and forgets the rest.3. It **refuses to promote** a retrained model that loses more than 1% test accuracy. A   retraining endpoint that blindly overwrites production is a way to break a working model   from the browser.The cell below simulates a run so you can see the mechanics.

In [ ]:
import io, zipfilefrom PIL import Imagefrom src import database as db, retraindb.init_db()# Pretend a user uploaded 40 new images of two classesrecords = []for ci in [class_names.index(class_names[2]), class_names.index(class_names[7])]:    for j, i in enumerate(np.where(y_test_raw == ci)[0][:20]):        records.append((ci, class_names[ci], f"upload_{ci}_{j}.png", x_test_raw[i]))saved = db.save_uploads(records)print(f"{saved} images written to the database; {db.upload_stats()['pending_uploads']} pending")result = retrain.run_retraining(trigger="notebook", epochs=3)print(result["status"], "|", result["message"])

In [ ]:
runs = pd.DataFrame(db.list_runs())runs[["id", "status", "trigger", "n_new_samples", "n_replay_samples", "promoted", "model_version"]]

---## 9. Appendix — the 100 fine labelsFor completeness, the same architecture trained on the 100 fine classes. The accuracy dropis the whole reason the deployed model uses the coarse labels: at 32x32, telling a `leopard`from a `tiger` is close to the limit of what the pixels contain.Run this only if you have time; it is not required for the deployed system.

In [ ]:
# os.environ["LABEL_MODE"] = "fine"  -> restart the kernel, then rerun sections 3-6.# Record the numbers here for the README comparison table.